In [1]:
import pandas as pd

In [2]:
age = pd.read_json("../data/Age.jsonl", lines=True)
disability = pd.read_json("../data/Disability_status.jsonl", lines=True)
gender = pd.read_json("../data/Gender_identity.jsonl",lines=True)
nationality = pd.read_json("../data/Nationality.jsonl", lines=True)
pysical_appearance = pd.read_json("../data/Physical_appearance.jsonl", lines=True)
Race_ethnicity = pd.read_json("../data/Race_ethnicity.jsonl", lines=True)
Religion = pd.read_json("../data/Religion.jsonl", lines=True)
SES = pd.read_json("../data/SES.jsonl", lines=True) # socioeconomic status
Sexual_orientation = pd.read_json("../data/Sexual_orientation.jsonl", lines=True)

In [3]:
age["additional_metadata"][0]

{'subcategory': 'None',
 'stereotyped_groups': ['old'],
 'version': 'a',
 'source': 'https://longevity.stanford.edu/older-adults-and-technology-moving-beyond-the-stereotypes/'}

In [4]:
age[:10]["answer_info"][0]['ans1']

['grandson', 'nonOld']

In [5]:
age["question_index"].value_counts()

question_index
23    336
25    336
19    288
21    280
22    280
16    200
18    200
10    160
20    160
24    160
6     128
7     128
8     128
14    128
4      96
11     96
12     96
17     96
5      72
9      72
13     72
15     72
1      32
2      32
3      32
Name: count, dtype: int64

In [6]:
label_groups_age = pd.json_normalize(age["answer_info"]).map(lambda x: x[1])

In [7]:
age["answer_info"][0]

{'ans0': ['grandfather', 'old'],
 'ans1': ['grandson', 'nonOld'],
 'ans2': ["Can't be determined", 'unknown']}

In [8]:
pd.json_normalize(age["additional_metadata"])["stereotyped_groups"].explode().unique().tolist()

['old', 'nonOld']

In [9]:
pd.json_normalize(disability["additional_metadata"])["stereotyped_groups"].explode().unique().tolist()

['disabled',
 'physically disabled',
 'mentally-ill',
 'D/deaf',
 "Down's syndrome",
 'people with cognitive disabilities or mental illness',
 'autistic people',
 'people with blindness or low-vision',
 'people with cerebral palsy']

In [12]:
# 원본 카테고리 이름을 유지해 수동 매핑표를 만든다.
datasets = {
    "age": age,
    "disability": disability,
    "gender": gender,
    "nationality": nationality,
    "pysical_appearance": pysical_appearance,
    "Race_ethnicity": Race_ethnicity,
    "Religion": Religion,
    "SES": SES,
    "Sexual_orientation": Sexual_orientation,
}

def ordered_unique(values):
    return pd.Series(values).dropna().drop_duplicates().tolist()

# stereotyped_group -> answer_info의 두 번째 값 수동 매핑표
stereotyped_group_to_answer_info = pd.concat(
    [
        pd.DataFrame({
            "category": category,
            "stereotyped_group": ordered_unique(
                pd.json_normalize(frame["additional_metadata"])["stereotyped_groups"].explode()
            ),
        })
        for category, frame in datasets.items()
    ],
    ignore_index=True,
).assign(answer_info_value=pd.NA)

# 매핑에 사용할 수 있는 category별 answer_info 후보 목록
answer_info_values_reference = pd.concat(
    [
        pd.DataFrame({
            "category": category,
            "answer_info_value": ordered_unique(
                pd.json_normalize(frame["answer_info"]).map(lambda x: x[1]).to_numpy().ravel()
            ),
        })
        for category, frame in datasets.items()
    ],
    ignore_index=True,
)

display(stereotyped_group_to_answer_info)
display(answer_info_values_reference)

,category,stereotyped_group,answer_info_value
0,age,old,<NA>
1,age,nonOld,<NA>
2,disability,disabled,<NA>
3,disability,physically disabled,<NA>
4,disability,mentally-ill,<NA>
...,...,...,...
71,SES,low SES,<NA>
72,Sexual_orientation,gay,<NA>
73,Sexual_orientation,bisexual,<NA>
74,Sexual_orientation,lesbian,<NA>


,category,answer_info_value
0,age,old
1,age,nonOld
2,age,unknown
3,disability,nonDisabled
4,disability,unknown
...,...,...
79,Sexual_orientation,unknown
80,Sexual_orientation,gay
81,Sexual_orientation,straight
82,Sexual_orientation,bisexual


In [13]:
stereotyped_group_to_answer_info.to_csv(
    "stereotyped_group_to_answer_info_mapping.csv",
    index=False,
    encoding="utf-8-sig",
)

answer_info_values_reference.to_csv(
    "answer_info_values_reference.csv",
    index=False,
    encoding="utf-8-sig",
)